# 01 — Training Walkthrough

Train the DenseNet121 classifier on NIH ChestX-ray14.

**Covers**: data loading, model init, training loop, checkpoint saving.

> **T4 GPU (Kaggle/Colab)**: set `TRAINING_MODE=kaggle` or `TRAINING_MODE=full`
> **Laptop (CPU)**: set `TRAINING_MODE=subset` for a fast smoke-test

## 1. Setup

In [1]:
import sys, os
sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.13.0+cu130
CUDA available: True
GPU: Tesla T4


## 2. Configure Training Mode

Change `TRAINING_MODE` to match your environment.

In [2]:
# TRAINING_MODE is read from your .env file by default.
# Uncomment below to override for this notebook session specifically:
# os.environ['TRAINING_MODE'] = 'subset'   # 'subset' | 'full' | 'kaggle'
# os.environ['SUBSET_SIZE']   = '500'
# os.environ['TRAIN_EPOCHS']  = '3'

from src.utils.config import TrainingConfig
print(f'Mode        : {TrainingConfig.mode}')
print(f'Subset size : {TrainingConfig.subset_size}')
print(f'Epochs      : {TrainingConfig.epochs}')
print(f'Batch size  : {TrainingConfig.batch_size()}')
print(f'Device      : {"cuda" if torch.cuda.is_available() else "cpu"}')

Mode        : full
Subset size : 500
Epochs      : 20
Batch size  : 32
Device      : cuda


## 3. Load Data

In [3]:
from src.data.extract import prepare_data
from src.data.dataloader import build_dataloaders

labels_df, image_dir = prepare_data()

# subset_size caps the total images used regardless of source - only
# apply it in 'subset' mode, otherwise use every image that's available.
subset_size = TrainingConfig.subset_size if TrainingConfig.mode == 'subset' else 0

train_loader, val_loader, test_loader = build_dataloaders(
    labels_df, image_dir,
    subset_size=subset_size,
)
print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

2026-07-14 18:14:03 | INFO     | src.data.extract                    | Loading NIH labels from /teamspace/studios/this_studio/chest-xray-classifier/data/labels/Data_Entry_2017.csv
2026-07-14 18:14:03 | INFO     | src.data.extract                    | Labels loaded: 112120 images, 40081 with at least one pathology
2026-07-14 18:14:03 | INFO     | src.data.extract                    | Validating images in /teamspace/studios/this_studio/chest-xray-classifier/data/raw/images against 112120 labels...
2026-07-14 18:14:06 | WARNING  | src.data.extract                    | 87121 images missing from disk — excluded
2026-07-14 18:14:06 | INFO     | src.data.extract                    | 24999 valid images available for training
2026-07-14 18:14:07 | INFO     | src.data.dataloader                 | No official splits found — using random 70/10/20 split
2026-07-14 18:14:07 | INFO     | src.data.dataloader                 | Random split — train: 17499, val: 2500, test: 5000
2026-07-14 18:14:09 | INF

## 4. Run Training

In [4]:
from src.training.train import train

# dry_run=True runs a 1-batch smoke test so this notebook stays fast/safe
# to run top-to-bottom. Set dry_run=False for a real training run (can take
# a long time in 'full' mode - the CLI `python -m src.training.train` is
# the better tool for that; use this cell mainly to inspect the training loop).
results = train(
    resume  = False,
    dry_run = True,
)
print(f"Best val AUC: {results['best_auc']:.4f}")

2026-07-14 18:14:17 | INFO     | src.training.train                  | ============================================================
2026-07-14 18:14:17 | INFO     | src.training.train                  |   CHEST X-RAY CLASSIFIER — TRAINING
2026-07-14 18:14:17 | INFO     | src.training.train                  |   mode=full  backbone=densenet121  epochs=20
2026-07-14 18:14:17 | INFO     | src.training.train                  | ============================================================
2026-07-14 18:14:17 | INFO     | src.training.train                  | Device: cuda
2026-07-14 18:14:17 | INFO     | src.training.train                  | GPU: Tesla T4 (15.6 GB VRAM)
2026-07-14 18:14:17 | INFO     | src.training.train                  | Loading data (mode=full)...
2026-07-14 18:14:17 | INFO     | src.data.extract                    | Loading NIH labels from /teamspace/studios/this_studio/chest-xray-classifier/data/labels/Data_Entry_2017.csv
2026-07-14 18:14:18 | INFO     | src.data.extract 

## 5. Kaggle Full Training

To train on the full NIH dataset using Kaggle's free T4 GPU:

1. Upload this notebook to Kaggle
2. Add dataset: `nih-chest-xrays` (pre-mounted)
3. Set `TRAINING_MODE=kaggle` and `TRAIN_EPOCHS=10`
4. Run all cells
5. Download `best_model.pt` from the output tab

In [ ]:
# Kaggle-specific setup
import os
if os.path.exists('/kaggle/input'):
    os.environ['TRAINING_MODE'] = 'kaggle'
    os.environ['TRAIN_EPOCHS']  = '10'
    print('Running on Kaggle T4 GPU')
else:
    print('Not on Kaggle — using subset mode')